In [ ]:
#Qué hace: integra todas las piezas en un dataset diario unificado alineado a T₀.
#Clave: input final para análisis/modelado; dataset esparso por diseño.

In [1]:
# ============================================================
# 09_dataset_diario_DIANA_unificado.ipynb
# ------------------------------------------------------------
# Objetivo:
# - Unir dominios fisiológicos (07) + labs (08) a nivel día
# - Mantener "measured flags" para no confundir missing con "no OK"
# - Construir endpoints diarios tipo DIANA:
#     physio_ok_observed
#     labs_ok_observed
#     diana_daily_ok_observed
#
# Entradas:
#   - 07_dominios_fisiologia.parquet
#   - 08_labs_diarios.parquet
#
# Salidas:
#   - 09_diana_daily.parquet
# ============================================================

In [2]:
import pandas as pd
import numpy as np

In [3]:
# -----------------------------
# 1) Cargar inputs
# -----------------------------
phys_path = "07_dominios_clinicos.parquet"
labs_path = "08_labs_diarios.parquet"

df_phys = pd.read_parquet(phys_path)
df_labs = pd.read_parquet(labs_path)

print("df_phys:", df_phys.shape)
print("df_labs:", df_labs.shape)

# Claves
KEYS = ["subject_id", "hadm_id", "icu_stay_id", "day_idx"]

missing_phys = set(KEYS) - set(df_phys.columns)
missing_labs = set(KEYS) - set(df_labs.columns)
if missing_phys:
    raise ValueError(f"Missing keys in df_phys: {missing_phys}")
if missing_labs:
    raise ValueError(f"Missing keys in df_labs: {missing_labs}")

df_phys: (126791, 13)
df_labs: (170806, 12)


In [4]:
# -----------------------------
# 2) Normalizar tipos claves
# -----------------------------
for k in KEYS:
    # day_idx int, el resto int64
    if k == "day_idx":
        df_phys[k] = df_phys[k].astype(int)
        df_labs[k] = df_labs[k].astype(int)
    else:
        df_phys[k] = df_phys[k].astype("int64")
        df_labs[k] = df_labs[k].astype("int64")



In [5]:
# -----------------------------
# 3) Merge (outer para no perder días)
# -----------------------------
df = df_phys.merge(df_labs, on=KEYS, how="outer", suffixes=("", "_labs"))

df = df.sort_values(KEYS).reset_index(drop=True)
print("Merged df:", df.shape)

Merged df: (177591, 21)


In [6]:
# -----------------------------
# 4) Helper: AND "observed" (ignora NaN)
# -----------------------------
def and_observed(*cols: pd.Series) -> pd.Series:
    """
    AND lógico ignorando NaNs:
    - Si todas son NaN -> NaN
    - Si alguna (no-NaN) es 0 -> 0
    - Si todas las no-NaN son 1 -> 1
    """
    mat = pd.concat(cols, axis=1)
    any_obs = mat.notna().any(axis=1)
    any_zero = (mat == 0).any(axis=1)
    all_one_among_obs = (mat.fillna(1) == 1).all(axis=1)  # NaN tratados como neutro
    out = np.where(~any_obs, np.nan, np.where(any_zero, 0, np.where(all_one_among_obs, 1, np.nan)))
    return pd.Series(out, index=mat.index)

In [7]:
# -----------------------------
# 5) Identificar columnas del 07 (robusto)
# -----------------------------
# Esperamos que existan (según tu 07): temp_ok, map_ok, sf_ok
# Si en tu 07 se llaman distinto, ajusta aquí.
expected_phys = ["temp_ok", "map_ok", "sf_ok"]
for c in expected_phys:
    if c not in df.columns:
        raise ValueError(f"Column '{c}' not found in merged df. Check your 07 output columns.")

# Flags de medida (si existen). Si no existen, los creamos a partir del valor.
if "temp_measured" not in df.columns:
    df["temp_measured"] = df["Temp"].notna().astype(int) if "Temp" in df.columns else df["temp_ok"].notna().astype(int)
if "map_measured" not in df.columns:
    df["map_measured"] = df["MAP"].notna().astype(int) if "MAP" in df.columns else df["map_ok"].notna().astype(int)
if "sf_measured" not in df.columns:
    df["sf_measured"] = df["SF_ratio"].notna().astype(int) if "SF_ratio" in df.columns else df["sf_ok"].notna().astype(int)

In [8]:
# -----------------------------
# 6) physio_ok_observed
# -----------------------------
# Definición conservadora:
# - calculamos AND observado de temp_ok, map_ok, sf_ok
# - si no hay ninguna medida relevante -> NaN
df["physio_ok_observed"] = and_observed(df["temp_ok"], df["map_ok"], df["sf_ok"])

# También puedes querer una versión "all_measured" estricta
df["physio_all_measured"] = ((df["temp_measured"] == 1) & (df["map_measured"] == 1) & (df["sf_measured"] == 1)).astype(int)

In [9]:
# -----------------------------
# 7) labs_ok_observed
# -----------------------------
# Usamos:
# - wbc_in_range (si existe) o, si no, lo creamos a partir de WBC
# - lactate_low (ya lo creaste en 08)
if "wbc_in_range" not in df.columns:
    if "WBC" not in df.columns:
        raise ValueError("Need either 'wbc_in_range' or 'WBC' in labs to build labs_ok_observed.")
    df["wbc_in_range"] = np.where(
        df["WBC"].notna(),
        ((df["WBC"] >= 4.0) & (df["WBC"] <= 12.0)).astype(int),
        np.nan
    )

if "lactate_low" not in df.columns:
    if "Lactate" not in df.columns:
        raise ValueError("Need either 'lactate_low' or 'Lactate' in labs to build labs_ok_observed.")
    df["lactate_low"] = np.where(
        df["Lactate"].notna(),
        (df["Lactate"] < 2.0).astype(int),
        np.nan
    )

# AND observado de (wbc_in_range, lactate_low)
df["labs_ok_observed"] = and_observed(df["wbc_in_range"], df["lactate_low"])

# Flags de medida
df["wbc_measured"] = df["WBC"].notna().astype(int) if "WBC" in df.columns else df["wbc_in_range"].notna().astype(int)
df["lactate_measured"] = df["Lactate"].notna().astype(int) if "Lactate" in df.columns else df["lactate_low"].notna().astype(int)

In [10]:
# -----------------------------
# 8) DIANA daily composite
# -----------------------------
# Definición propuesta (conservadora):
# - DIANA ok si physio_ok_observed == 1 AND labs_ok_observed == 1,
#   pero como labs puede faltar mucho (lactato), hacemos AND observado global.
df["diana_daily_ok_observed"] = and_observed(df["physio_ok_observed"], df["labs_ok_observed"])

In [11]:
# -----------------------------
# 9) QA rápido
# -----------------------------
print("\nRates (notna):")
for c in ["physio_ok_observed", "labs_ok_observed", "diana_daily_ok_observed"]:
    print(c, "observed rate:", df[c].notna().mean(), "| ok among observed:", df[c].mean(skipna=True))

print("\nMissingness summary:")
for c in ["temp_ok","map_ok","sf_ok","wbc_in_range","lactate_low"]:
    print(c, "NaN rate:", df[c].isna().mean())


Rates (notna):
physio_ok_observed observed rate: 0.7119899093985619 | ok among observed: 0.5748360921521951
labs_ok_observed observed rate: 0.9617942350682186 | ok among observed: 0.4830568012833273
diana_daily_ok_observed observed rate: 0.9985697473408 | ok among observed: 0.36872733834450794

Missingness summary:
temp_ok NaN rate: 0.31274670450642206
map_ok NaN rate: 0.30656958967515247
sf_ok NaN rate: 0.5122782122967943
wbc_in_range NaN rate: 0.04303709084356752
lactate_low NaN rate: 0.6801977577692563


In [12]:
# -----------------------------
# 10) Guardar
# -----------------------------
out_path = "09_diana_daily.parquet"
df.to_parquet(out_path, index=False)
print("Saved:", out_path)

df.head(10)

Saved: 09_diana_daily.parquet


,subject_id,hadm_id,icu_stay_id,day_idx,Temp,MAP,SpO2,FiO2,FiO2_frac,SF_ratio_clean,...,wbc_in_range,wbc_trend,wbc_improving,temp_measured,map_measured,sf_measured,physio_ok_observed,physio_all_measured,labs_ok_observed,diana_daily_ok_observed
0,10001217,24597018,37067082,0,36.972222,90.0,98.0,NaN,NaN,NaN,...,0.0,NaN,NaN,1,1,0,1.0,0,0.0,0.0
1,10001217,24597018,37067082,1,37.611111,94.0,95.0,NaN,NaN,NaN,...,0.0,1.0,1.0,1,1,0,1.0,0,0.0,0.0
2,10002428,20321825,34807493,0,36.944444,63.0,100.0,25.0,0.25,400.0,...,0.0,NaN,NaN,1,1,1,0.0,1,0.0,0.0
3,10002428,20321825,34807493,1,37.027778,67.5,97.0,NaN,NaN,NaN,...,1.0,1.0,1.0,1,1,0,1.0,0,1.0,1.0
4,10002428,23473524,35479615,0,36.666667,82.0,99.0,NaN,NaN,NaN,...,1.0,NaN,NaN,1,1,0,1.0,0,1.0,1.0
5,10002428,23473524,35479615,1,36.694444,81.5,99.0,NaN,NaN,NaN,...,1.0,0.0,0.0,1,1,0,1.0,0,1.0,1.0
6,10002428,23473524,35479615,2,36.944444,91.0,99.0,NaN,NaN,NaN,...,NaN,NaN,NaN,1,1,0,1.0,0,NaN,1.0
7,10002428,28662225,33987268,0,36.583333,68.0,100.0,NaN,NaN,NaN,...,0.0,NaN,NaN,1,1,0,1.0,0,0.0,0.0
8,10002428,28662225,33987268,1,36.583333,68.5,98.0,NaN,NaN,NaN,...,0.0,1.0,1.0,1,1,0,1.0,0,0.0,0.0
9,10002428,28662225,33987268,2,36.555556,64.5,96.5,NaN,NaN,NaN,...,0.0,1.0,1.0,1,1,0,0.0,0,0.0,0.0
